In [1]:
!pip install -q "transformers<5" sacrebleu nltk rouge-score bert-score scikit-learn scipy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os, re, gc, json, glob, unicodedata, difflib, warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "/kaggle/working/extra_metrics"
os.makedirs(OUT_DIR, exist_ok=True)

ENABLE_METEOR = True
ENABLE_ROUGE = True
ENABLE_BERTSCORE = True
ENABLE_NER = True
ENABLE_INTENT = True
N_INTENT = 100
INTENT_LABELS = ["a question", "a decision or agreement", "a request or action item",
                 "a schedule or deadline", "a factual statement", "a greeting or closing remark"]
LANG_NAMES = {"hi": "Hindi", "te": "Telugu", "ta": "Tamil", "bn": "Bengali", "kn": "Kannada",
              "ml": "Malayalam", "mr": "Marathi", "gu": "Gujarati", "pa": "Punjabi", "or": "Odia"}

def find_results_dir():
    cands = ["/kaggle/working/results_multilingual"]
    cands += glob.glob("/kaggle/input/**/results_multilingual", recursive=True)
    for c in cands:
        if os.path.isdir(c):
            return c
    hits = glob.glob("/kaggle/input/**/mt_df.csv", recursive=True) + glob.glob("/kaggle/input/**/cascade_df.csv", recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    raise FileNotFoundError("results_multilingual folder not found. Run the main notebook first, "
                            "or attach its results zip as a dataset.")

RESULTS_DIR = find_results_dir()
LANGS = sorted(d for d in os.listdir(RESULTS_DIR)
               if os.path.isdir(os.path.join(RESULTS_DIR, d))
               and (os.path.exists(os.path.join(RESULTS_DIR, d, "mt_df.csv"))
                    or os.path.exists(os.path.join(RESULTS_DIR, d, "cascade_df.csv"))))
print("results dir:", RESULTS_DIR, "| languages found:", LANGS)

def load_lang(l):
    d = os.path.join(RESULTS_DIR, l)
    out = {}
    for k in ("mt_df", "cascade_df", "asr_df", "stream_df"):
        p = os.path.join(d, f"{k}.csv")
        out[k] = pd.read_csv(p).fillna("") if os.path.exists(p) else None
    return out

DATA = {l: load_lang(l) for l in LANGS}
for l, d in DATA.items():
    print(l, {k: (len(v) if v is not None else None) for k, v in d.items()})

results dir: /kaggle/input/datasets/alwalmedhana/results-baseline | languages found: ['ta', 'te']
ta {'mt_df': 200, 'cascade_df': 50, 'asr_df': 100, 'stream_df': 17}
te {'mt_df': 200, 'cascade_df': 50, 'asr_df': 100, 'stream_df': 20}


In [3]:
def safe(fn, name):
    try:
        return fn()
    except Exception as e:
        print(f"  [{name} skipped: {type(e).__name__}: {str(e)[:120]}]")
        return None

def meteor_score_mean(hyps, refs):
    import nltk
    from nltk.translate.meteor_score import meteor_score
    for pkg in ("wordnet", "omw-1.4"):
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
    sc = [meteor_score([r.lower().split()], h.lower().split()) for h, r in zip(hyps, refs)]
    return 100 * float(np.mean(sc))

def rouge_scores(hyps, refs):
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r = [scorer.score(ref, hyp) for hyp, ref in zip(hyps, refs)]
    return {name: 100 * float(np.mean([x[k].fmeasure for x in r]))
            for name, k in (("ROUGE-1", "rouge1"), ("ROUGE-2", "rouge2"), ("ROUGE-L", "rougeL"))}

def bertscore_scores(hyps, refs):
    from bert_score import score
    P, R, F = score([h if h.strip() else "." for h in hyps], refs, lang="en", device=device,
                    batch_size=16, verbose=False)
    return {"BERTScore_P": 100 * P.mean().item(), "BERTScore_R": 100 * R.mean().item(),
            "BERTScore_F1": 100 * F.mean().item()}

def text_metrics(hyps, refs):
    import sacrebleu
    from sacrebleu.metrics import CHRF
    hyps, refs = list(map(str, hyps)), list(map(str, refs))
    m = {"BLEU": sacrebleu.corpus_bleu(hyps, [refs]).score,
         "chrF++": CHRF(word_order=2).corpus_score(hyps, [refs]).score}
    if ENABLE_METEOR:
        v = safe(lambda: meteor_score_mean(hyps, refs), "METEOR")
        if v is not None:
            m["METEOR"] = v
    if ENABLE_ROUGE:
        v = safe(lambda: rouge_scores(hyps, refs), "ROUGE")
        if v:
            m.update(v)
    if ENABLE_BERTSCORE:
        v = safe(lambda: bertscore_scores(hyps, refs), "BERTScore")
        if v:
            m.update(v)
    return m

In [4]:
UNITS = {w: i for i, w in enumerate("zero one two three four five six seven eight nine".split())}
TEENS = {w: 10 + i for i, w in enumerate(
    "ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split())}
TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70,
        "eighty": 80, "ninety": 90}
SCALES = {"thousand": 1000, "lakh": 100000, "lakhs": 100000, "million": 1000000, "crore": 10000000}
WEEKDAYS = {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}
MONTHS = {"january", "february", "march", "april", "may", "june", "july", "august", "september",
          "october", "november", "december"}

def to_ascii_digits(text):
    return "".join(str(unicodedata.digit(ch)) if unicodedata.digit(ch, None) is not None else ch for ch in text)

def _kind(w):
    if w in UNITS: return "unit"
    if w in TEENS: return "teen"
    if w in TENS: return "tens"
    if w == "hundred": return "hund"
    if w in SCALES: return "scale"
    return None

def _can_follow(kind, last):
    if kind == "unit":
        return last in ("tens", "hund", "scale")
    if kind in ("teen", "tens"):
        return last in ("hund", "scale")
    if kind == "hund":
        return last in ("unit", "teen", "tens")
    if kind == "scale":
        return last in ("unit", "teen", "tens", "hund")
    return False

def words_to_numbers(text):
    toks = re.findall(r"[A-Za-z]+|\d+(?:[.,:]\d+)*|\s+|.", text, flags=re.S)
    out = []
    st = dict(total=0, cur=0, last=None, count=0, pending="")

    def reset():
        st.update(total=0, cur=0, last=None, count=0, pending="")

    def flush():
        if st["count"]:
            val = st["total"] + st["cur"]
            out.append("one" if (st["count"] == 1 and val == 1) else str(val))
            out.append(st["pending"])
        reset()

    def apply(kind, w):
        if kind in ("unit", "teen", "tens"):
            st["cur"] += UNITS.get(w, TEENS.get(w, TENS.get(w, 0)))
        elif kind == "hund":
            st["cur"] = (st["cur"] or 1) * 100
        else:
            st["total"] += (st["cur"] or 1) * SCALES[w]
            st["cur"] = 0
        st["count"] += 1
        st["last"] = kind
        st["pending"] = ""

    for t in toks:
        w = t.lower()
        if t.isspace():
            if st["count"]:
                st["pending"] += t
            else:
                out.append(t)
            continue
        if w == "and" and st["count"] and st["last"] in ("hund", "scale"):
            st["pending"] += t
            continue
        kind = _kind(w)
        if kind and (st["count"] == 0 or _can_follow(kind, st["last"])):
            apply(kind, w)
        else:
            flush()
            if kind:
                apply(kind, w)
            else:
                out.append(t)
    flush()
    return "".join(out)

def canon(text):
    t = words_to_numbers(to_ascii_digits(str(text)))
    return re.sub(r"(\d)\.(\d{2})\b", r"\1:\2", t)

def numeric_entities(text):
    t = canon(text)
    ents = [("TIME", m.group()) for m in re.finditer(r"\b\d{1,2}:\d{2}\b", t)]
    t2 = re.sub(r"\b\d{1,2}:\d{2}\b", " ", t)
    ents += [("NUM", m.group().replace(",", "")) for m in re.finditer(r"\d+(?:,\d{3})*(?:\.\d+)?", t2)]
    ents += [("DATE", w) for w in re.findall(r"[a-z]+", t.lower()) if w in WEEKDAYS or w in MONTHS]
    return ents

ner_pipe = None
if ENABLE_NER:
    try:
        from transformers import pipeline
        ner_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple",
                            device=0 if device == "cuda" else -1)
    except Exception as e:
        print("NER model unavailable, numeric/date entities only:", repr(e)[:150])

_ner_cache = {}

def ner_entities(text):
    if ner_pipe is None or not str(text).strip():
        return []
    if text not in _ner_cache:
        res = []
        for e in ner_pipe(str(text)):
            w = e["word"].strip()
            if len(w) > 1 and float(e.get("score", 1.0)) >= 0.7:
                res.append(({"PER": "PERSON"}.get(e["entity_group"], e["entity_group"]), w))
        _ner_cache[text] = res
    return _ner_cache[text]

def all_entities(text):
    return numeric_entities(text) + ner_entities(text)

def _tokens(s):
    return re.findall(r"[a-z0-9:]+", canon(s).lower())

def entity_in(ent_text, ent_type, hyp_text):
    ht, et = _tokens(hyp_text), _tokens(ent_text)
    if not et:
        return False
    n = len(et)
    fuzzy = ent_type in ("PERSON", "ORG", "LOC", "MISC")
    for i in range(len(ht) - n + 1):
        win = ht[i:i + n]
        if win == et:
            return True
        if fuzzy and difflib.SequenceMatcher(None, " ".join(win), " ".join(et)).ratio() >= 0.8:
            return True
    return False

def entity_scores(refs, hyps):
    rec_hit, rec_tot, pre_hit, pre_tot = {}, {}, {}, {}
    sent_with, sent_all = 0, 0
    for ref, hyp in zip(refs, hyps):
        r_ents, h_ents = all_entities(ref), all_entities(hyp)
        hits = 0
        for typ, e in r_ents:
            ok = entity_in(e, typ, hyp)
            rec_tot[typ] = rec_tot.get(typ, 0) + 1
            rec_hit[typ] = rec_hit.get(typ, 0) + int(ok)
            hits += int(ok)
        if r_ents:
            sent_with += 1
            sent_all += int(hits == len(r_ents))
        for typ, e in h_ents:
            pre_tot[typ] = pre_tot.get(typ, 0) + 1
            pre_hit[typ] = pre_hit.get(typ, 0) + int(entity_in(e, typ, ref))
    rows = []
    for typ in sorted(set(rec_tot) | set(pre_tot)):
        rc = rec_hit.get(typ, 0) / rec_tot[typ] if rec_tot.get(typ) else np.nan
        pr = pre_hit.get(typ, 0) / pre_tot[typ] if pre_tot.get(typ) else np.nan
        f1 = 2 * rc * pr / (rc + pr) if (rc == rc and pr == pr and rc + pr > 0) else np.nan
        rows.append(dict(type=typ, ref_entities=rec_tot.get(typ, 0), recall=100 * rc,
                         hyp_entities=pre_tot.get(typ, 0), precision=100 * pr, f1=100 * f1))
    R, RT = sum(rec_hit.values()), sum(rec_tot.values())
    P, PT = sum(pre_hit.values()), sum(pre_tot.values())
    rc = R / RT if RT else np.nan
    pr = P / PT if PT else np.nan
    f1 = 2 * rc * pr / (rc + pr) if (RT and PT and rc + pr > 0) else np.nan
    overall = {"ENT_recall_%": 100 * rc, "ENT_precision_%": 100 * pr, "ENT_F1_%": 100 * f1,
               "ENT_ref_entities": RT,
               "ENT_all_preserved_sentences_%": 100 * sent_all / sent_with if sent_with else np.nan}
    return overall, pd.DataFrame(rows)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


In [5]:
intent_clf = None
if ENABLE_INTENT:
    try:
        from transformers import pipeline
        intent_clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",
                              device=0 if device == "cuda" else -1)
    except Exception as e:
        print("Intent classifier unavailable:", repr(e)[:150])

def classify_intents(texts):
    out = []
    for i in range(0, len(texts), 16):
        batch = [t if str(t).strip() else "." for t in texts[i:i + 16]]
        res = intent_clf(batch, candidate_labels=INTENT_LABELS, hypothesis_template="This sentence is {}.")
        res = res if isinstance(res, list) else [res]
        out += [r["labels"][0] for r in res]
    return out

def intent_agreement(refs, hyps, n=N_INTENT):
    from sklearn.metrics import cohen_kappa_score
    if intent_clf is None:
        return {}
    refs, hyps = list(refs)[:n], list(hyps)[:n]
    a, b = classify_intents(refs), classify_intents(hyps)
    agree = float(np.mean([x == y for x, y in zip(a, b)]))
    kappa = float(cohen_kappa_score(a, b)) if len(set(a) | set(b)) > 1 else np.nan
    return {"INTENT_agreement_%": 100 * agree, "INTENT_kappa": kappa, "INTENT_n": len(a)}

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [6]:
def sentence_chrf(hyps, refs):
    import sacrebleu
    return [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(hyps, refs)]

def score_block(prefix, hyps, refs, extra_wer=None):
    m, ent_df = {}, None
    tm = text_metrics(hyps, refs)
    m.update({f"{prefix}_{k}": v for k, v in tm.items()})
    ov, ent_df = entity_scores(refs, hyps)
    m.update({f"{prefix}_{k}": v for k, v in ov.items()})
    if intent_clf is not None:
        m.update({f"{prefix}_{k}": v for k, v in intent_agreement(refs, hyps).items()})
    if extra_wer is not None and len(hyps) > 5:
        from scipy.stats import spearmanr
        rho, p = spearmanr(extra_wer, sentence_chrf(hyps, refs))
        m[f"{prefix}_spearman_WER_vs_chrF"] = float(rho)
    return m, ent_df

RESULTS = {}
for l in LANGS:
    print("=" * 60, "\n", LANG_NAMES.get(l, l))
    d, m, ents = DATA[l], {}, {}
    if d["mt_df"] is not None and len(d["mt_df"]):
        mt = d["mt_df"]
        mm, ents["MT"] = score_block("MT", mt.hypothesis.tolist(), mt.reference.tolist())
        m.update(mm)
    if d["cascade_df"] is not None and len(d["cascade_df"]):
        c = d["cascade_df"]
        cm, ents["CASCADE"] = score_block("CASCADE", c.asr_english.tolist(), c.gold_english.tolist(),
                                          extra_wer=c.asr_wer.astype(float).tolist())
        m.update(cm)
    RESULTS[l] = dict(metrics=m, entities=ents)
    os.makedirs(os.path.join(OUT_DIR, l), exist_ok=True)
    for k, df in ents.items():
        if df is not None:
            df.to_csv(os.path.join(OUT_DIR, l, f"entity_by_type_{k}.csv"), index=False)
    print({k: round(v, 2) if isinstance(v, float) else v for k, v in m.items()})

 Tamil


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'MT_BLEU': 32.59, 'MT_chrF++': 57.7, 'MT_METEOR': 55.68, 'MT_ROUGE-1': 66.87, 'MT_ROUGE-2': 40.91, 'MT_ROUGE-L': 58.79, 'MT_BERTScore_P': 94.55, 'MT_BERTScore_R': 94.55, 'MT_BERTScore_F1': 94.55, 'MT_ENT_recall_%': 74.32, 'MT_ENT_precision_%': 74.74, 'MT_ENT_F1_%': 74.53, 'MT_ENT_ref_entities': 584, 'MT_ENT_all_preserved_sentences_%': 52.0, 'MT_INTENT_agreement_%': 86.0, 'MT_INTENT_kappa': 0.77, 'MT_INTENT_n': 100, 'CASCADE_BLEU': 62.27, 'CASCADE_chrF++': 76.21, 'CASCADE_METEOR': 78.11, 'CASCADE_ROUGE-1': 82.15, 'CASCADE_ROUGE-2': 70.12, 'CASCADE_ROUGE-L': 77.54, 'CASCADE_BERTScore_P': 96.11, 'CASCADE_BERTScore_R': 96.54, 'CASCADE_BERTScore_F1': 96.32, 'CASCADE_ENT_recall_%': 66.67, 'CASCADE_ENT_precision_%': 60.0, 'CASCADE_ENT_F1_%': 63.16, 'CASCADE_ENT_ref_entities': 45, 'CASCADE_ENT_all_preserved_sentences_%': 60.0, 'CASCADE_INTENT_agreement_%': 82.0, 'CASCADE_INTENT_kappa': 0.71, 'CASCADE_INTENT_n': 50, 'CASCADE_spearman_WER_vs_chrF': -0.68}
 Telugu


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'MT_BLEU': 40.17, 'MT_chrF++': 63.39, 'MT_METEOR': 62.21, 'MT_ROUGE-1': 73.18, 'MT_ROUGE-2': 49.04, 'MT_ROUGE-L': 64.83, 'MT_BERTScore_P': 95.58, 'MT_BERTScore_R': 95.4, 'MT_BERTScore_F1': 95.48, 'MT_ENT_recall_%': 78.08, 'MT_ENT_precision_%': 80.03, 'MT_ENT_F1_%': 79.05, 'MT_ENT_ref_entities': 584, 'MT_ENT_all_preserved_sentences_%': 56.0, 'MT_INTENT_agreement_%': 82.0, 'MT_INTENT_kappa': 0.71, 'MT_INTENT_n': 100, 'CASCADE_BLEU': 62.46, 'CASCADE_chrF++': 76.51, 'CASCADE_METEOR': 77.33, 'CASCADE_ROUGE-1': 83.42, 'CASCADE_ROUGE-2': 69.06, 'CASCADE_ROUGE-L': 80.48, 'CASCADE_BERTScore_P': 96.54, 'CASCADE_BERTScore_R': 96.66, 'CASCADE_BERTScore_F1': 96.6, 'CASCADE_ENT_recall_%': 78.18, 'CASCADE_ENT_precision_%': 72.73, 'CASCADE_ENT_F1_%': 75.36, 'CASCADE_ENT_ref_entities': 55, 'CASCADE_ENT_all_preserved_sentences_%': 70.97, 'CASCADE_INTENT_agreement_%': 92.0, 'CASCADE_INTENT_kappa': 0.84, 'CASCADE_INTENT_n': 50, 'CASCADE_spearman_WER_vs_chrF': -0.46}


In [8]:
MEETING_KEYWORDS = [
    ["three", "3"], ["progress report", "progress"], ["deadline", "time limit"], ["next meeting"],
    ["two teams", "two groups"], ["design"], ["testing", "test"], ["objection", "objections"],
    ["budget"], ["finance department", "finance"], ["friday"], ["10", "ten"], ["zoom"],
    ["presentation"], ["email", "e-mail"], ["thank", "thanks"],
]

def keyword_recall(text, keywords=MEETING_KEYWORDS):
    t = canon(text).lower()
    found, missing = 0, []
    for variants in keywords:
        if any(re.search(r"\b" + re.escape(v.lower()) + r"\b", t) for v in variants):
            found += 1
        else:
            missing.append(variants[0])
    return 100 * found / len(keywords), missing

paths = ["/kaggle/working/transcript.csv"] + glob.glob("/kaggle/input/**/transcript.csv", recursive=True)
tp = next((p for p in paths if os.path.exists(p)), None)
if tp:
    tr = pd.read_csv(tp).fillna("")
    rec, missing = keyword_recall(" ".join(tr["english"].astype(str)))
    print(f"Meeting keyword recall: {rec:.1f}%  ({len(MEETING_KEYWORDS) - len(missing)}/{len(MEETING_KEYWORDS)})")
    print("Missing:", missing)
    MEETING_RECALL = rec
else:
    print("transcript.csv not found - skipping (it is written by the Hindi meeting run of the earlier notebook).")
    MEETING_RECALL = None

transcript.csv not found - skipping (it is written by the Hindi meeting run of the earlier notebook).


In [9]:
import shutil

summary = pd.DataFrame({LANG_NAMES.get(l, l): pd.Series(r["metrics"]) for l, r in RESULTS.items()})
summary.to_csv(os.path.join(OUT_DIR, "extra_metrics_summary.csv"))
pd.set_option("display.max_rows", 200)
display(summary.round(2))

for l, r in RESULTS.items():
    for k, df in r["entities"].items():
        if df is not None and len(df):
            print(f"\n{LANG_NAMES.get(l, l)} - entity preservation by type ({k})")
            display(df.round(1))

if MEETING_RECALL is not None:
    print(f"\nOwn meeting recording keyword recall: {MEETING_RECALL:.1f}%")
shutil.make_archive("/kaggle/working/extra_metrics", "zip", OUT_DIR)
print("Saved /kaggle/working/extra_metrics.zip")

,Tamil,Telugu
MT_BLEU,32.59,40.17
MT_chrF++,57.70,63.39
MT_METEOR,55.68,62.21
MT_ROUGE-1,66.87,73.18
MT_ROUGE-2,40.91,49.04
MT_ROUGE-L,58.79,64.83
MT_BERTScore_P,94.55,95.58
MT_BERTScore_R,94.55,95.40
MT_BERTScore_F1,94.55,95.48
MT_ENT_recall_%,74.32,78.08



Tamil - entity preservation by type (MT)


,type,ref_entities,recall,hyp_entities,precision,f1
0,DATE,25,96.0,24,95.8,95.9
1,LOC,112,79.5,103,80.6,80.0
2,MISC,110,77.3,103,78.6,78.0
3,NUM,151,78.1,157,79.6,78.9
4,ORG,68,82.4,60,86.7,84.5
5,PERSON,111,50.5,121,48.8,49.6
6,TIME,7,85.7,6,100.0,92.3



Tamil - entity preservation by type (CASCADE)


,type,ref_entities,recall,hyp_entities,precision,f1
0,DATE,4,75.0,3,100.0,85.7
1,LOC,11,81.8,9,66.7,73.5
2,MISC,8,87.5,7,100.0,93.3
3,NUM,4,50.0,4,50.0,50.0
4,ORG,5,60.0,4,75.0,66.7
5,PERSON,13,46.2,18,33.3,38.7



Telugu - entity preservation by type (MT)


,type,ref_entities,recall,hyp_entities,precision,f1
0,DATE,25,92.0,24,95.8,93.9
1,LOC,112,83.9,112,90.2,86.9
2,MISC,110,80.0,96,84.4,82.1
3,NUM,151,82.1,156,82.1,82.1
4,ORG,68,89.7,62,93.5,91.6
5,PERSON,111,55.0,121,53.7,54.3
6,TIME,7,71.4,5,100.0,83.3



Telugu - entity preservation by type (CASCADE)


,type,ref_entities,recall,hyp_entities,precision,f1
0,DATE,2,100.0,2,100.0,100.0
1,LOC,12,58.3,11,63.6,60.9
2,MISC,6,100.0,6,100.0,100.0
3,NUM,9,88.9,8,100.0,94.1
4,ORG,9,88.9,7,100.0,94.1
5,PERSON,17,70.6,21,47.6,56.9


Saved /kaggle/working/extra_metrics.zip
